<a href="https://colab.research.google.com/github/NhanChanDeo/VQA_Final_Thesis/blob/main/DATASETS%20SCHEMA%20NORMALIZATION/Step6_Normalize_CrossVQA_Parquet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 6: End-to-End Dataset Normalization & Parquet Conversion (CrossVQA)

This notebook normalizes **TextVQA** (`gamusa/textvqa-v0.5.1`) and **DocVQA** (`gamusa/docvqa-task1-spdocvqa`) into the **Canonical CrossVQA Schema**:
1. **Harmonized Splits:** Standardizes to `train`, `validation`, and `test`.
2. **Spatial Normalization:** Scales all OCR bounding boxes (Rosetta & Microsoft OCR) to `[ymin, xmin, ymax, xmax] ∈ [0, 1000]`.
3. **Visual Quality:** Auto-corrects EXIF rotation for TextVQA.
4. **Parquet Serialization:** Serializes into columnar Parquet files with embedded images for instant Hugging Face Dataset Viewer browsing.
5. **Hub Deployment:** Uploads canonical datasets to Hugging Face.

In [1]:
# 1. Environment Setup & HF Authentication
!pip install -q --upgrade huggingface_hub datasets pyarrow pandas tqdm Pillow

import os
import io
import json
import zipfile
import tarfile
from collections import Counter
from pathlib import Path
from typing import List, Dict, Any, Optional
from PIL import Image, ImageOps
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm
from huggingface_hub import HfApi, login, hf_hub_download

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.getenv('HF_TOKEN')

if not HF_TOKEN:
    raise ValueError("Please configure HF_TOKEN in Colab Secrets or environment variables!")

login(token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)
username = api.whoami()['name']
print(f"✓ Authenticated as: {username}")

# Workspace paths
WORK_DIR = Path("./crossvqa_workspace")
RAW_DIR = WORK_DIR / "raw"
NORM_DIR = WORK_DIR / "normalized"
RAW_DIR.mkdir(parents=True, exist_ok=True)
NORM_DIR.mkdir(parents=True, exist_ok=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 988.2 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.0/838.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 80.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
tokenizers 0.23.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.1.1 which 

In [2]:
# 2. Canonical Helper Functions
def extract_canonical_answer(answers: Optional[List[str]]) -> Optional[str]:
    if not answers:
        return None
    cleaned = [ans.strip() for ans in answers if ans and ans.strip()]
    if not cleaned:
        return None
    return Counter(cleaned).most_common(1)[0][0]

def normalize_rosetta_box(box_dict: Dict[str, float]) -> List[int]:
    tl_x = box_dict.get("top_left_x", 0.0)
    tl_y = box_dict.get("top_left_y", 0.0)
    w = box_dict.get("width", 0.0)
    h = box_dict.get("height", 0.0)
    xmin = int(round(max(0.0, min(1.0, tl_x)) * 1000))
    ymin = int(round(max(0.0, min(1.0, tl_y)) * 1000))
    xmax = int(round(max(0.0, min(1.0, tl_x + w)) * 1000))
    ymax = int(round(max(0.0, min(1.0, tl_y + h)) * 1000))
    return [ymin, xmin, ymax, xmax]

def normalize_ms_ocr_polygon(polygon: List[float], img_w: int, img_h: int) -> List[int]:
    if not polygon or len(polygon) < 8 or img_w <= 0 or img_h <= 0:
        return [0, 0, 0, 0]
    xs = polygon[0::2]
    ys = polygon[1::2]
    xmin = int(round(max(0.0, min(1.0, min(xs) / img_w)) * 1000))
    xmax = int(round(max(0.0, min(1.0, max(xs) / img_w)) * 1000))
    ymin = int(round(max(0.0, min(1.0, min(ys) / img_h)) * 1000))
    ymax = int(round(max(0.0, min(1.0, max(ys) / img_h)) * 1000))
    return [ymin, xmin, ymax, xmax]

In [3]:
# 3. Download TextVQA Assets from Hub
TEXTVQA_REPO = f"{username}/textvqa-v0.5.1"
print(f"Downloading TextVQA assets from {TEXTVQA_REPO}...")

textvqa_files = [
    "annotations/TextVQA_0.5.1_train.json",
    "annotations/TextVQA_0.5.1_val.json",
    "annotations/TextVQA_0.5.1_test.json",
    "ocr/TextVQA_Rosetta_OCR_v0.2_train.json",
    "ocr/TextVQA_Rosetta_OCR_v0.2_val.json",
    "ocr/TextVQA_Rosetta_OCR_v0.2_test.json",
    "images/train_val_images.zip",
    "images/test_images.zip",
]

textvqa_paths = {}
for rfile in textvqa_files:
    textvqa_paths[rfile] = hf_hub_download(
        repo_id=TEXTVQA_REPO,
        repo_type="dataset",
        filename=rfile,
        local_dir=str(RAW_DIR / "textvqa")
    )
print("✓ TextVQA assets downloaded successfully.")

annotations/TextVQA_0.5.1_train.json: reconstructing file:   0%|          |  0.00B / 21.6MB            

annotations/TextVQA_0.5.1_train.json: downloading bytes:           |  0.00B            

TextVQA_0.5.1_val.json:   0%|          | 0.00/3.12M [00:00<?, ?B/s]

TextVQA_0.5.1_test.json:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

ocr/TextVQA_Rosetta_OCR_v0.2_train.json: reconstructing file:   0%|          |  0.00B / 65.8MB            

...

images/train_val_images.zip: reconstructing file:   0%|          |  0.00B / 7.07GB            

images/train_val_images.zip: downloading bytes:           |  0.00B            

images/test_images.zip: reconstructing file:   0%|          |  0.00B /  970MB            

images/test_images.zip: downloading bytes:           |  0.00B            

✓ TextVQA assets downloaded successfully.


In [4]:
# 4. Normalize and Export TextVQA to Parquet
def process_textvqa(split_name, ann_key, ocr_key, zip_key):
    ann_path = textvqa_paths[ann_key]
    ocr_path = textvqa_paths[ocr_key]
    zip_path = textvqa_paths[zip_key]
    out_path = NORM_DIR / f"textvqa_{split_name}.parquet"

    with open(ann_path, 'r', encoding='utf-8') as f:
        items = json.load(f)['data']
    with open(ocr_path, 'r', encoding='utf-8') as f:
        ocr_raw = json.load(f)['data']

    ocr_lookup = {}
    for o in ocr_raw:
        tokens = o.get('ocr_tokens', [])
        boxes = [normalize_rosetta_box(b.get('bounding_box', {})) for b in o.get('ocr_info', [])]
        ocr_lookup[o['image_id']] = {'tokens': tokens, 'boxes': boxes}

    zip_ref = zipfile.ZipFile(zip_path, 'r')
    prefix = "train_images/" if split_name != "test" else "test_images/"

    rows = []
    for item in tqdm(items, desc=f"TextVQA [{split_name}]"):
        q_id = str(item['question_id'])
        img_id = item['image_id']
        img_name = f"{prefix}{img_id}.jpg"

        raw_bytes = zip_ref.read(img_name)
        img = ImageOps.exif_transpose(Image.open(io.BytesIO(raw_bytes)))
        orig_w, orig_h = img.size

        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=95)

        ocr_entry = ocr_lookup.get(img_id, {'tokens': [], 'boxes': []})
        answers = item.get('answers', None)

        rows.append({
            "sample_id": f"textvqa_{q_id}",
            "domain_type": "scene_text",
            "split": split_name,
            "question_id": q_id,
            "question": item['question'].strip(),
            "question_tokens": item.get('question_tokens', []),
            "image_id": img_id,
            "image": {'bytes': buf.getvalue(), 'path': f"{img_id}.jpg"},
            "image_width": orig_w,
            "image_height": orig_h,
            "answers": answers if answers else [],
            "canonical_answer": extract_canonical_answer(answers),
            "ocr_tokens": ocr_entry['tokens'],
            "ocr_boxes": ocr_entry['boxes'],
            "metadata": json.dumps({
                "image_classes": item.get('image_classes', []),
                "flickr_original_url": item.get('flickr_original_url', '')
            }, ensure_ascii=False)
        })
    zip_ref.close()

    df = pd.DataFrame(rows)
    df.to_parquet(out_path, index=False, engine='pyarrow', compression='zstd')
    print(f"✓ Exported {out_path} ({len(df)} rows, {out_path.stat().st_size / (1024**2):.1f} MB)")

process_textvqa("train", "annotations/TextVQA_0.5.1_train.json", "ocr/TextVQA_Rosetta_OCR_v0.2_train.json", "images/train_val_images.zip")
process_textvqa("validation", "annotations/TextVQA_0.5.1_val.json", "ocr/TextVQA_Rosetta_OCR_v0.2_val.json", "images/train_val_images.zip")
process_textvqa("test", "annotations/TextVQA_0.5.1_test.json", "ocr/TextVQA_Rosetta_OCR_v0.2_test.json", "images/test_images.zip")

TextVQA [train]:   0%|          | 0/34602 [00:00<?, ?it/s]

✓ Exported crossvqa_workspace/normalized/textvqa_train.parquet (34602 rows, 4808.8 MB)


TextVQA [validation]:   0%|          | 0/5000 [00:00<?, ?it/s]

✓ Exported crossvqa_workspace/normalized/textvqa_validation.parquet (5000 rows, 709.0 MB)


TextVQA [test]:   0%|          | 0/5734 [00:00<?, ?it/s]

✓ Exported crossvqa_workspace/normalized/textvqa_test.parquet (5734 rows, 727.0 MB)


In [5]:
# 5. Download DocVQA Assets from Hub
DOCVQA_REPO = f"{username}/docvqa-task1-spdocvqa"
print(f"Downloading DocVQA assets from {DOCVQA_REPO}...")

docvqa_files = [
    "annotations/train_v1.0_withQT.json",
    "annotations/val_v1.0_withQT.json",
    "annotations/test_v1.0.json",
    "images/spdocvqa_images.tar.gz",
    "ocr/spdocvqa_ocr.tar.gz"
]

docvqa_paths = {}
for rfile in docvqa_files:
    docvqa_paths[rfile] = hf_hub_download(
        repo_id=DOCVQA_REPO,
        repo_type="dataset",
        filename=rfile,
        local_dir=str(RAW_DIR / "docvqa")
    )
print("✓ DocVQA assets downloaded successfully.")

annotations/train_v1.0_withQT.json: reconstructing file:   0%|          |  0.00B / 11.6MB            

annotations/train_v1.0_withQT.json: downloading bytes:           |  0.00B            

val_v1.0_withQT.json:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

test_v1.0.json:   0%|          | 0.00/1.15M [00:00<?, ?B/s]

images/spdocvqa_images.tar.gz: reconstructing file:   0%|          |  0.00B / 8.58GB            

images/spdocvqa_images.tar.gz: downloading bytes:           |  0.00B            

ocr/spdocvqa_ocr.tar.gz: reconstructing file:   0%|          |  0.00B / 62.9MB            

ocr/spdocvqa_ocr.tar.gz: downloading bytes:           |  0.00B            

✓ DocVQA assets downloaded successfully.


In [10]:
# 6. Normalize and Export DocVQA to Parquet
def process_docvqa(split_name, ann_key):
    ann_path = docvqa_paths[ann_key]
    img_tar_path = docvqa_paths["images/spdocvqa_images.tar.gz"]
    ocr_tar_path = docvqa_paths["ocr/spdocvqa_ocr.tar.gz"]
    out_path = NORM_DIR / f"docvqa_{split_name}.parquet"

    with open(ann_path, 'r', encoding='utf-8') as f:
        items = json.load(f)['data']

    print("Indexing Tar archives for instant O(1) lookups...")
    img_tar = tarfile.open(img_tar_path, 'r:*')
    # Map basename to member object for O(1) retrieval
    img_members = {}
    for member in img_tar.getmembers():
        if member.isfile():
            img_members[os.path.basename(member.name)] = member
            img_members[member.name.lstrip("./")] = member

    ocr_tar = tarfile.open(ocr_tar_path, 'r:*')
    ocr_members = {}
    for member in ocr_tar.getmembers():
        if member.isfile():
            ocr_members[os.path.basename(member.name)] = member
            ocr_members[member.name.lstrip("./")] = member

    rows = []
    for item in tqdm(items, desc=f"DocVQA Optimized [{split_name}]"):
        q_id = str(item['questionId'])
        img_rel = item['image'].lstrip("./")
        img_basename = os.path.basename(img_rel)

        # O(1) Image Extraction
        member = img_members.get(img_rel) or img_members.get(img_basename)
        if not member:
            continue
        f_img = img_tar.extractfile(member)
        raw_bytes = f_img.read()

        img = Image.open(io.BytesIO(raw_bytes))
        orig_w, orig_h = img.size

        # Speedup: Save as high-quality JPEG instead of heavy, lossless PNG
        buf = io.BytesIO()
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")
        img.save(buf, format="JPEG", quality=95)

        # O(1) OCR Extraction
        base_id = os.path.splitext(img_basename)[0]
        ocr_tokens, ocr_boxes = [], []
        for c_path in [f"ocr_results/{base_id}.json", f"{base_id}.json"]:
            ocr_member = ocr_members.get(c_path) or ocr_members.get(f"{base_id}.json")
            if ocr_member:
                f_ocr = ocr_tar.extractfile(ocr_member)
                ocr_json = json.loads(f_ocr.read().decode('utf-8'))
                for page in ocr_json.get('recognitionResults', []):
                    for line in page.get('lines', []):
                        for w in line.get('words', []):
                            t = w.get('text', '').strip()
                            poly = w.get('boundingBox', [])
                            if t and poly:
                                ocr_tokens.append(t)
                                ocr_boxes.append(normalize_ms_ocr_polygon(poly, orig_w, orig_h))
                break

        answers = item.get('answers', None)
        rows.append({
            "sample_id": f"docvqa_{q_id}",
            "domain_type": "document_text",
            "split": split_name,
            "question_id": q_id,
            "question": item['question'].strip(),
            "question_tokens": item['question'].lower().split(),
            "image_id": img_basename,
            "image": {'bytes': buf.getvalue(), 'path': img_basename},
            "image_width": orig_w,
            "image_height": orig_h,
            "answers": answers if answers else [],
            "canonical_answer": extract_canonical_answer(answers),
            "ocr_tokens": ocr_tokens,
            "ocr_boxes": ocr_boxes,
            "metadata": json.dumps({
                "question_types": item.get('question_types', []),
                "docId": item.get('docId', None),
                "ucsf_document_id": item.get('ucsf_document_id', ''),
                "ucsf_document_page_no": item.get('ucsf_document_page_no', '')
            }, ensure_ascii=False)
        })

    img_tar.close()
    ocr_tar.close()

    df = pd.DataFrame(rows)
    df.to_parquet(out_path, index=False, engine='pyarrow', compression='snappy')
    print(f"✓ Exported {out_path} ({len(df)} rows, {out_path.stat().st_size / (1024**2):.1f} MB)")

process_docvqa("train", "annotations/train_v1.0_withQT.json")
process_docvqa("validation", "annotations/val_v1.0_withQT.json")
process_docvqa("test", "annotations/test_v1.0.json")

Indexing Tar archives for instant O(1) lookups...


DocVQA Optimized [train]:   0%|          | 0/39463 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [11]:
import io
import json
import os
import subprocess
from pathlib import Path
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

# 1. Sequential linear extraction to disk (takes ~1 min total for 8.5 GB)
DOC_IMG_DIR = RAW_DIR / "docvqa_extracted_images"
DOC_OCR_DIR = RAW_DIR / "docvqa_extracted_ocr"
DOC_IMG_DIR.mkdir(parents=True, exist_ok=True)
DOC_OCR_DIR.mkdir(parents=True, exist_ok=True)

if not any(DOC_IMG_DIR.iterdir()):
  print("Extracting images sequentially to disk (one-time linear pass)...")
  subprocess.run(
      [
          "tar",
          "-xzf",
          str(docvqa_paths["images/spdocvqa_images.tar.gz"]),
          "-C",
          str(DOC_IMG_DIR),
      ],
      check=True,
  )

if not any(DOC_OCR_DIR.iterdir()):
  print("Extracting OCR sequentially to disk (one-time linear pass)...")
  subprocess.run(
      [
          "tar",
          "-xzf",
          str(docvqa_paths["ocr/spdocvqa_ocr.tar.gz"]),
          "-C",
          str(DOC_OCR_DIR),
      ],
      check=True,
  )


# 2. Optimized Processing Function
def process_docvqa(split_name, ann_key):
  ann_path = docvqa_paths[ann_key]
  out_path = NORM_DIR / f"docvqa_{split_name}.parquet"

  with open(ann_path, "r", encoding="utf-8") as f:
    items = json.load(f)["data"]

  # Caches so shared pages are only read & processed once
  image_cache = {}  # img_basename -> {'bytes': ..., 'w': ..., 'h': ...}
  ocr_cache = {}  # base_id -> (tokens, boxes)

  rows = []
  for item in tqdm(items, desc=f"DocVQA [{split_name}]"):
    q_id = str(item["questionId"])
    img_rel = item["image"].lstrip("./")
    img_basename = os.path.basename(img_rel)
    base_id = os.path.splitext(img_basename)[0]

    # --- A. Fast Cached Image Retrieval ---
    if img_basename not in image_cache:
      # Locate file on disk
      img_disk_path = DOC_IMG_DIR / img_rel
      if not img_disk_path.exists():
        img_disk_path = DOC_IMG_DIR / img_basename
        if not img_disk_path.exists():
          continue

      with Image.open(img_disk_path) as img:
        orig_w, orig_h = img.size
        if img.mode in ("RGBA", "P"):
          img = img.convert("RGB")

        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=90, optimize=True)

      image_cache[img_basename] = {
          "bytes": buf.getvalue(),
          "w": orig_w,
          "h": orig_h,
      }

    cached_img = image_cache[img_basename]

    # --- B. Fast Cached OCR Retrieval ---
    if base_id not in ocr_cache:
      ocr_tokens, ocr_boxes = [], []
      ocr_candidate = DOC_OCR_DIR / f"ocr_results/{base_id}.json"
      if not ocr_candidate.exists():
        ocr_candidate = DOC_OCR_DIR / f"{base_id}.json"

      if ocr_candidate.exists():
        with open(ocr_candidate, "r", encoding="utf-8") as f_ocr:
          ocr_json = json.load(f_ocr)
          for page in ocr_json.get("recognitionResults", []):
            for line in page.get("lines", []):
              for w in line.get("words", []):
                t = w.get("text", "").strip()
                poly = w.get("boundingBox", [])
                if t and poly:
                  ocr_tokens.append(t)
                  ocr_boxes.append(
                      normalize_ms_ocr_polygon(
                          poly, cached_img["w"], cached_img["h"]
                      )
                  )
      ocr_cache[base_id] = (ocr_tokens, ocr_boxes)

    ocr_tokens, ocr_boxes = ocr_cache[base_id]
    answers = item.get("answers", None)

    rows.append({
        "sample_id": f"docvqa_{q_id}",
        "domain_type": "document_text",
        "split": split_name,
        "question_id": q_id,
        "question": item["question"].strip(),
        "question_tokens": item["question"].lower().split(),
        "image_id": img_basename,
        "image": {"bytes": cached_img["bytes"], "path": img_basename},
        "image_width": cached_img["w"],
        "image_height": cached_img["h"],
        "answers": answers if answers else [],
        "canonical_answer": extract_canonical_answer(answers),
        "ocr_tokens": ocr_tokens,
        "ocr_boxes": ocr_boxes,
        "metadata": json.dumps(
            {
                "question_types": item.get("question_types", []),
                "docId": item.get("docId", None),
                "ucsf_document_id": item.get("ucsf_document_id", ""),
                "ucsf_document_page_no": item.get("ucsf_document_page_no", ""),
            },
            ensure_ascii=False,
        ),
    })

  df = pd.DataFrame(rows)
  df.to_parquet(out_path, index=False, engine="pyarrow", compression="snappy")
  print(
      f"✓ Exported {out_path} ({len(df)} rows,"
      f" {out_path.stat().st_size / (1024**2):.1f} MB)"
  )


process_docvqa("train", "annotations/train_v1.0_withQT.json")
process_docvqa("validation", "annotations/val_v1.0_withQT.json")
process_docvqa("test", "annotations/test_v1.0.json")

Extracting images sequentially to disk (one-time linear pass)...
Extracting OCR sequentially to disk (one-time linear pass)...


DocVQA [train]:   0%|          | 0/39463 [00:00<?, ?it/s]

✓ Exported crossvqa_workspace/normalized/docvqa_train.parquet (39463 rows, 18682.7 MB)


DocVQA [validation]:   0%|          | 0/5349 [00:00<?, ?it/s]

✓ Exported crossvqa_workspace/normalized/docvqa_validation.parquet (5349 rows, 2221.1 MB)


DocVQA [test]:   0%|          | 0/5188 [00:00<?, ?it/s]

✓ Exported crossvqa_workspace/normalized/docvqa_test.parquet (5188 rows, 2230.6 MB)


In [14]:
# 7. Verification of Normalized Splits (Optimized using PyArrow Metadata)
import pyarrow.parquet as pq

def verify_parquet_split_count(filepath, expected_count):
    # Reads only the Parquet file metadata (O(1) time), completely avoiding image decoding errors
    pf = pq.ParquetFile(filepath)
    actual_count = pf.metadata.num_rows
    assert actual_count == expected_count, f"Count mismatch for {filepath.name}: expected {expected_count}, got {actual_count}"
    print(f"  {filepath.name}: {actual_count} rows verified.")

print("=== VERIFYING TEXTVQA NORMALIZED SPLITS ===")
try:
    verify_parquet_split_count(NORM_DIR / 'textvqa_train.parquet', 34602)
    verify_parquet_split_count(NORM_DIR / 'textvqa_validation.parquet', 5000)
    verify_parquet_split_count(NORM_DIR / 'textvqa_test.parquet', 5734)
    print("✓ TextVQA split counts verified: 100% Match!")
except Exception as e:
    print(f"❌ TextVQA verification failed: {e}")

print("\n=== VERIFYING DOCVQA NORMALIZED SPLITS ===")
try:
    verify_parquet_split_count(NORM_DIR / 'docvqa_train.parquet', 39463)
    verify_parquet_split_count(NORM_DIR / 'docvqa_validation.parquet', 5349)
    verify_parquet_split_count(NORM_DIR / 'docvqa_test.parquet', 5188)
    print("✓ DocVQA split counts verified: 100% Match!")
except Exception as e:
    print(f"❌ DocVQA verification failed: {e}")

=== VERIFYING TEXTVQA NORMALIZED SPLITS ===
  textvqa_train.parquet: 34602 rows verified.
  textvqa_validation.parquet: 5000 rows verified.
  textvqa_test.parquet: 5734 rows verified.
✓ TextVQA split counts verified: 100% Match!

=== VERIFYING DOCVQA NORMALIZED SPLITS ===
  docvqa_train.parquet: 39463 rows verified.
  docvqa_validation.parquet: 5349 rows verified.
  docvqa_test.parquet: 5188 rows verified.
✓ DocVQA split counts verified: 100% Match!


In [15]:
# 7.1. Inspect the folder structures within crossvqa_workspace
import os

def print_dir_tree(startpath, max_depth=3):
    for root, dirs, files in os.walk(startpath):
        # Calculate current depth relative to startpath
        depth = root.replace(startpath, '').count(os.sep)
        if depth > max_depth:
            continue
        indent = ' ' * 4 * depth
        print(f"{indent}📁 {os.path.basename(root)}/")
        sub_indent = ' ' * 4 * (depth + 1)
        # Show up to 5 files to prevent cluttering the log
        for f in files[:5]:
            print(f"{sub_indent}📄 {f}")
        if len(files) > 5:
            print(f"{sub_indent}📄 ... and {len(files) - 5} more files")

print_dir_tree('./crossvqa_workspace')

📁 crossvqa_workspace/
    📁 normalized/
        📄 docvqa_test.parquet
        📄 docvqa_validation.parquet
        📄 textvqa_train.parquet
        📄 docvqa_train.parquet
        📄 textvqa_test.parquet
        📄 ... and 1 more files
    📁 raw/
        📁 docvqa_extracted_ocr/
            📄 fkxn0226_26.json
            📄 ltgj0223_13.json
            📄 zlfv0228_20.json
            📄 klhv0228_2.json
            📄 qtvg0227_2.json
            📄 ... and 12762 more files
        📁 docvqa/
            📁 annotations/
                📄 train_v1.0_withQT.json
                📄 test_v1.0.json
                📄 val_v1.0_withQT.json
            📁 .cache/
            📁 images/
                📄 spdocvqa_images.tar.gz
            📁 ocr/
                📄 spdocvqa_ocr.tar.gz
        📁 textvqa/
            📁 annotations/
                📄 TextVQA_0.5.1_val.json
                📄 TextVQA_0.5.1_train.json
                📄 TextVQA_0.5.1_test.json
            📁 .cache/
            📁 images/
                📄 

In [16]:
# 8. Upload Canonical Repositories to Hugging Face
def upload_canonical_repo(repo_name, prefix, task_category, tags):
    full_repo = f"{username}/{repo_name}"
    api.create_repo(repo_id=full_repo, repo_type="dataset", exist_ok=True)

    # Write README.md with Dataset Info
    readme_content = f"""---
configs:
- config_name: default
  data_files:
  - split: train
    path: data/train-*\n  - split: validation
    path: data/validation-*\n  - split: test
    path: data/test-*\ntask_categories:
- {task_category}
tags:
{tags}
---
# {repo_name}
Normalized Canonical Dataset for CrossVQA Benchmark.\n"""
    readme_path = WORK_DIR / f"README_{repo_name}.md"
    readme_path.write_text(readme_content, encoding='utf-8')
    api.upload_file(
        path_or_fileobj=str(readme_path),
        path_in_repo="README.md",
        repo_id=full_repo,
        repo_type="dataset"
    )

    # Upload Parquet files
    for split in ["train", "validation", "test"]:
        p_file = NORM_DIR / f"{prefix}_{split}.parquet"
        api.upload_file(
            path_or_fileobj=str(p_file),
            path_in_repo=f"data/{split}-00000.parquet",
            repo_id=full_repo,
            repo_type="dataset"
        )
    print(f"✓ Successfully uploaded: https://huggingface.co/datasets/{full_repo}")

# Upload TextVQA Canonical
upload_canonical_repo(
    repo_name="textvqa-canonical",
    prefix="textvqa",
    task_category="visual-question-answering",
    tags="- textvqa\n- scene-text\n- canonical\n- crossvqa"
)

# Upload DocVQA Canonical
upload_canonical_repo(
    repo_name="docvqa-canonical",
    prefix="docvqa",
    task_category="visual-question-answering",
    tags="- docvqa\n- document-ai\n- canonical\n- crossvqa"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...zed/textvqa_train.parquet:   0%|          |  524kB / 5.04GB            

Validating                    :           |   0%            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...lized/docvqa_test.parquet:   0%|          |  551kB / 2.34GB            

Validating                    :           |   0%            

✓ Successfully uploaded: https://huggingface.co/datasets/gamusa/docvqa-canonical


In [29]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

NORM_DIR = Path("./crossvqa_workspace/normalized")
file_path = NORM_DIR / "docvqa_validation.parquet"

print(f"\n{'='*20} INSPECTING: {file_path.name} {'='*20}")

# 1. See PyArrow Data Schema
parquet_file = pq.ParquetFile(file_path)
print("\n--- PyArrow Data Schema ---")
print(parquet_file.schema)

# 2. Read table and combine chunks to safely handle nested/complex columns

table = table.combine_chunks()
df = table.to_pandas(safe=False)

# 3. See Pandas Info
print("\n--- DataFrame Info ---")
df.info()

# 4. Preview first 10 entries
print("\n--- First 10 Entries (Preview) ---")
preview_df = df.copy()
if 'image' in preview_df.columns:
    preview_df['image'] = preview_df['image'].apply(lambda x: f"Image (bytes size: {len(x['bytes']) if isinstance(x, dict) and 'bytes' in x else 'Unknown'})" if x else None)

display(preview_df.head(10))


==================== INSPECTING: docvqa_validation.parquet ====================

--- PyArrow Data Schema ---
required group field_id=-1 schema {
  optional binary field_id=-1 sample_id (String);
  optional binary field_id=-1 domain_type (String);
  optional binary field_id=-1 split (String);
  optional binary field_id=-1 question_id (String);
  optional binary field_id=-1 question (String);
  optional group field_id=-1 question_tokens (List) {
    repeated group field_id=-1 list {
      optional binary field_id=-1 element (String);
    }
  }
  optional binary field_id=-1 image_id (String);
  optional group field_id=-1 image {
    optional binary field_id=-1 bytes;
    optional binary field_id=-1 path (String);
  }
  optional int64 field_id=-1 image_width;
  optional int64 field_id=-1 image_height;
  optional group field_id=-1 answers (List) {
    repeated group field_id=-1 list {
      optional binary field_id=-1 element (String);
    }
  }
  optional binary field_id=-1 canonical_answ

,sample_id,domain_type,split,question_id,question,question_tokens,image_id,image,image_width,image_height,answers,canonical_answer,ocr_tokens,ocr_boxes,metadata
0,textvqa_34602,scene_text,validation,34602,what is the brand of this camera?,"[what, is, the, brand, of, this, camera]",003a8ae2ef43b901,Image (bytes size: 87675),1024,664,"[nous les gosses, dakota, clos culombu, dakota...",dakota,"[DAKOTA, DIGITAL, Single-Use, Camera, Pire, di...","[[156, 148, 225, 261], [181, 265, 241, 373], [...","{""image_classes"": [""Cassette deck"", ""Printer"",..."
1,textvqa_34603,scene_text,validation,34603,what does the small white text spell?,"[what, does, the, small, white, text, spell]",b9dc400eb20bad64,Image (bytes size: 226037),1024,683,"[copenhagen, copenhagen, copenhagen, copenhage...",copenhagen,"[D, RUPALCON, PALCON, COPENHAGEN]","[[164, 100, 304, 193], [173, 199, 434, 934], [...","{""image_classes"": [""Billboard""], ""flickr_origi..."
2,textvqa_34604,scene_text,validation,34604,what kind of beer is this?,"[what, kind, of, beer, is, this]",2b538a43dd933fc1,Image (bytes size: 235913),1024,1024,"[ale, sublimely self-righteous ale, stone, ale...",ale,"[NINK, NK, BOWING, CC, STON, SUE, ED, Sublimel...","[[123, 97, 171, 208], [123, 146, 173, 206], [1...","{""image_classes"": [""Surfboard"", ""Drink"", ""Bott..."
3,textvqa_34605,scene_text,validation,34605,what brand liquor is on the right?,"[what, brand, liquor, is, on, the, right]",831bcec304a17054,Image (bytes size: 248327),1024,576,"[bowmore , bowmore, bowmore, bowmore, bowmore,...",bowmore,"[IGNET, N, MORANGIE, MORANGIE, MORANGIE, CHLAN...","[[96, 310, 155, 438], [98, 351, 155, 377], [24...","{""image_classes"": [""Beer"", ""Drink"", ""Bottle"", ..."
4,textvqa_34606,scene_text,validation,34606,how long has the drink on the right been aged?,"[how, long, has, the, drink, on, the, right, b...",831bcec304a17054,Image (bytes size: 248327),1024,576,"[10 years, 10 year, 10 years, 10 years , 10 ye...",10 years,"[IGNET, N, MORANGIE, MORANGIE, MORANGIE, CHLAN...","[[96, 310, 155, 438], [98, 351, 155, 377], [24...","{""image_classes"": [""Beer"", ""Drink"", ""Bottle"", ..."
5,textvqa_34607,scene_text,validation,34607,what number is on the player's jersey?,"[what, number, is, on, the, player, ', s, jersey]",94ad4aad01e27a32,Image (bytes size: 329528),1024,670,"[22, 22, 22, 22, 22, 22, 22, 22, 22, 22]",22,[22],"[[374, 475, 451, 530]]","{""image_classes"": [""Boy"", ""Person"", ""Man"", ""Sp..."
6,textvqa_34608,scene_text,validation,34608,what is the time?,"[what, is, the, time]",181f00d3ee2b2076,Image (bytes size: 125738),1024,729,"[5:41, 5:41, 8:00, 5:41, 5:40, 5:41, 5:42, 5:4...",5:41,"[N, u, g0]","[[228, 588, 280, 628], [285, 734, 323, 765], [...","{""image_classes"": [""Clock"", ""Fashion accessory..."
7,textvqa_34609,scene_text,validation,34609,what brand of watch is that?,"[what, brand, of, watch, is, that]",181f00d3ee2b2076,Image (bytes size: 125738),1024,729,"[ap, unanswerable, unanswerable, ap, af, unans...",unanswerable,"[N, u, g0]","[[228, 588, 280, 628], [285, 734, 323, 765], [...","{""image_classes"": [""Clock"", ""Fashion accessory..."
8,textvqa_34610,scene_text,validation,34610,who is at the center of all of this?,"[who, is, at, the, center, of, all, of, this]",30f9b48158fc56e6,Image (bytes size: 188938),1024,683,"[bryan owens, alexa curtis, bryan owens, bryan...",bryan owens,"[Aithell, BORATION, orysetdhe, Hayu, Cothone, ...","[[61, 535, 104, 577], [153, 2, 184, 54], [120,...","{""image_classes"": [""Whiteboard""], ""flickr_orig..."
9,textvqa_34611,scene_text,validation,34611,who was the photographer?,"[who, was, the, photographer]",4ffbb21f379cccbe,Image (bytes size: 146065),1024,681,"[philippe molitor, philippe molitor, philippe ...",philippe molitor,"[LAGOUDE, Gleamlight/PhilippeMolitor, Philippe]","[[675, 212, 715, 315], [965, 17, 984, 201], [9...","{""image_classes"": [""Person"", ""Drink""], ""flickr..."


In [33]:
import io
import pyarrow.parquet as pq
from pathlib import Path
from PIL import Image
import ipywidgets as widgets
from IPython.display import display

NORM_DIR = Path("./crossvqa_workspace/normalized")

def display_first_5_images(file_name):
    file_path = NORM_DIR / file_name
    print(f"\n{'='*20} First 5 Images from {file_name} {'='*20}")

    # Use iter_batches to read only the first batch of rows. This bypasses reading gigabytes of data
    # and avoids the 'chunked array' error completely.
    pfile = pq.ParquetFile(file_path)
    batch_iter = pfile.iter_batches(batch_size=5, columns=['image', 'question_id'])
    first_batch = next(batch_iter)

    # Convert only the small 5-row batch to a pandas dataframe
    df = first_batch.to_pandas()

    images = []
    titles = []

    for idx, row in df.iterrows():
        img_data = row['image']
        if isinstance(img_data, dict) and 'bytes' in img_data:
            img_bytes = img_data['bytes']
            img = Image.open(io.BytesIO(img_bytes))
            # Resize for visual consistency in preview
            img.thumbnail((250, 250))
            images.append(img)
            titles.append(f"Row {idx} ({row['question_id']})")

    # Display images side-by-side
    out_images = []
    for img, title in zip(images, titles):
        buf = io.BytesIO()
        img.save(buf, format="JPEG")
        img_widget = widgets.Image(value=buf.getvalue(), format='jpg', width=180, height=180)
        box = widgets.VBox([widgets.Label(value=title), img_widget])
        out_images.append(box)

    display(widgets.HBox(out_images))

# Display for both datasets
display_first_5_images("textvqa_validation.parquet")
display_first_5_images("docvqa_validation.parquet")


==================== First 5 Images from textvqa_validation.parquet ====================



==================== First 5 Images from docvqa_validation.parquet ====================


In [22]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

NORM_DIR = Path("./crossvqa_workspace/normalized")
file_path = NORM_DIR / "textvqa_validation.parquet"

print(f"\n{'='*20} INSPECTING: {file_path.name} {'='*20}")

# 1. See PyArrow Data Schema
parquet_file = pq.ParquetFile(file_path)
print("\n--- PyArrow Data Schema ---")
print(parquet_file.schema)

# 2. Read table and combine chunks to handle nested/complex data types
table = parquet_file.read()
table = table.combine_chunks()
df = table.to_pandas(safe=False)

# 3. See Pandas Info
print("\n--- DataFrame Info ---")
df.info()

# 4. Preview first 10 entries
print("\n--- First 10 Entries (Preview) ---")
preview_df = df.copy()
if 'image' in preview_df.columns:
    preview_df['image'] = preview_df['image'].apply(lambda x: f"Image (bytes size: {len(x['bytes']) if isinstance(x, dict) and 'bytes' in x else 'Unknown'})" if x else None)

display(preview_df.head(10))


==================== INSPECTING: textvqa_validation.parquet ====================

--- PyArrow Data Schema ---
required group field_id=-1 schema {
  optional binary field_id=-1 sample_id (String);
  optional binary field_id=-1 domain_type (String);
  optional binary field_id=-1 split (String);
  optional binary field_id=-1 question_id (String);
  optional binary field_id=-1 question (String);
  optional group field_id=-1 question_tokens (List) {
    repeated group field_id=-1 list {
      optional binary field_id=-1 element (String);
    }
  }
  optional binary field_id=-1 image_id (String);
  optional group field_id=-1 image {
    optional binary field_id=-1 bytes;
    optional binary field_id=-1 path (String);
  }
  optional int64 field_id=-1 image_width;
  optional int64 field_id=-1 image_height;
  optional group field_id=-1 answers (List) {
    repeated group field_id=-1 list {
      optional binary field_id=-1 element (String);
    }
  }
  optional binary field_id=-1 canonical_ans

,sample_id,domain_type,split,question_id,question,question_tokens,image_id,image,image_width,image_height,answers,canonical_answer,ocr_tokens,ocr_boxes,metadata
0,textvqa_34602,scene_text,validation,34602,what is the brand of this camera?,"[what, is, the, brand, of, this, camera]",003a8ae2ef43b901,Image (bytes size: 87675),1024,664,"[nous les gosses, dakota, clos culombu, dakota...",dakota,"[DAKOTA, DIGITAL, Single-Use, Camera, Pire, di...","[[156, 148, 225, 261], [181, 265, 241, 373], [...","{""image_classes"": [""Cassette deck"", ""Printer"",..."
1,textvqa_34603,scene_text,validation,34603,what does the small white text spell?,"[what, does, the, small, white, text, spell]",b9dc400eb20bad64,Image (bytes size: 226037),1024,683,"[copenhagen, copenhagen, copenhagen, copenhage...",copenhagen,"[D, RUPALCON, PALCON, COPENHAGEN]","[[164, 100, 304, 193], [173, 199, 434, 934], [...","{""image_classes"": [""Billboard""], ""flickr_origi..."
2,textvqa_34604,scene_text,validation,34604,what kind of beer is this?,"[what, kind, of, beer, is, this]",2b538a43dd933fc1,Image (bytes size: 235913),1024,1024,"[ale, sublimely self-righteous ale, stone, ale...",ale,"[NINK, NK, BOWING, CC, STON, SUE, ED, Sublimel...","[[123, 97, 171, 208], [123, 146, 173, 206], [1...","{""image_classes"": [""Surfboard"", ""Drink"", ""Bott..."
3,textvqa_34605,scene_text,validation,34605,what brand liquor is on the right?,"[what, brand, liquor, is, on, the, right]",831bcec304a17054,Image (bytes size: 248327),1024,576,"[bowmore , bowmore, bowmore, bowmore, bowmore,...",bowmore,"[IGNET, N, MORANGIE, MORANGIE, MORANGIE, CHLAN...","[[96, 310, 155, 438], [98, 351, 155, 377], [24...","{""image_classes"": [""Beer"", ""Drink"", ""Bottle"", ..."
4,textvqa_34606,scene_text,validation,34606,how long has the drink on the right been aged?,"[how, long, has, the, drink, on, the, right, b...",831bcec304a17054,Image (bytes size: 248327),1024,576,"[10 years, 10 year, 10 years, 10 years , 10 ye...",10 years,"[IGNET, N, MORANGIE, MORANGIE, MORANGIE, CHLAN...","[[96, 310, 155, 438], [98, 351, 155, 377], [24...","{""image_classes"": [""Beer"", ""Drink"", ""Bottle"", ..."
5,textvqa_34607,scene_text,validation,34607,what number is on the player's jersey?,"[what, number, is, on, the, player, ', s, jersey]",94ad4aad01e27a32,Image (bytes size: 329528),1024,670,"[22, 22, 22, 22, 22, 22, 22, 22, 22, 22]",22,[22],"[[374, 475, 451, 530]]","{""image_classes"": [""Boy"", ""Person"", ""Man"", ""Sp..."
6,textvqa_34608,scene_text,validation,34608,what is the time?,"[what, is, the, time]",181f00d3ee2b2076,Image (bytes size: 125738),1024,729,"[5:41, 5:41, 8:00, 5:41, 5:40, 5:41, 5:42, 5:4...",5:41,"[N, u, g0]","[[228, 588, 280, 628], [285, 734, 323, 765], [...","{""image_classes"": [""Clock"", ""Fashion accessory..."
7,textvqa_34609,scene_text,validation,34609,what brand of watch is that?,"[what, brand, of, watch, is, that]",181f00d3ee2b2076,Image (bytes size: 125738),1024,729,"[ap, unanswerable, unanswerable, ap, af, unans...",unanswerable,"[N, u, g0]","[[228, 588, 280, 628], [285, 734, 323, 765], [...","{""image_classes"": [""Clock"", ""Fashion accessory..."
8,textvqa_34610,scene_text,validation,34610,who is at the center of all of this?,"[who, is, at, the, center, of, all, of, this]",30f9b48158fc56e6,Image (bytes size: 188938),1024,683,"[bryan owens, alexa curtis, bryan owens, bryan...",bryan owens,"[Aithell, BORATION, orysetdhe, Hayu, Cothone, ...","[[61, 535, 104, 577], [153, 2, 184, 54], [120,...","{""image_classes"": [""Whiteboard""], ""flickr_orig..."
9,textvqa_34611,scene_text,validation,34611,who was the photographer?,"[who, was, the, photographer]",4ffbb21f379cccbe,Image (bytes size: 146065),1024,681,"[philippe molitor, philippe molitor, philippe ...",philippe molitor,"[LAGOUDE, Gleamlight/PhilippeMolitor, Philippe]","[[675, 212, 715, 315], [965, 17, 984, 201], [9...","{""image_classes"": [""Person"", ""Drink""], ""flickr..."
